# Two ways to write a speaker–listener model

A speaker sees three objects: **a green square, a green circle, and a pink circle**.
The speaker says **green**, **pink**, **square**, or **round**. Which object does the
listener infer? We compare the paper's memo model with an independent plain Python
Rational Speech Acts (RSA) model of the same task, then run both through UMI.

This demonstrates model integration and configuration comparisons. There is no
human-data fit or leaderboard score. The UMI dependency is unmerged PR #43. Its scoped regression qualification
is complete with Malania excluded; the broader release matrix remains separate. See [the report](../../docs/memo_umi_integration.md).

Use a **Python 3.11 UMI kernel** and set `MEMO_PYTHON` to the separate Python 3.12+
worker interpreter described in [the README](README.md). Run all cells in order.

In [1]:
import inspect
import json
import os
from pathlib import Path
import sys
import tempfile
from contextlib import contextmanager
from collections.abc import Iterator

import numpy as np
import pandas as pd
from IPython.display import Code, display
from brainscore_core.contract import Subject
from brainscore_core.streaming import BufferedSession, Session, StreamEvent
from brainscore.experiments import (
    Experiment, RecordInputsOutputs, SessionProtocol, Trial,
    compare_outputs, read_events, replay_sessions,
)
import brainscore.models.memo_rsa as memo_package
from brainscore.models.memo_rsa import MemoConfig, MemoSubject, MemoWorker
from brainscore.models.memo_rsa.model import CHOICES, SCENE, UTTERANCES

example_dir = Path.cwd()
if not (example_dir / "python_rsa.py").is_file():
    example_dir = example_dir / "examples" / "memo_umi"
if not (example_dir / "python_rsa.py").is_file():
    raise RuntimeError("Open from the repository root or examples/memo_umi")
sys.path.insert(0, str(example_dir))
from python_rsa import listener_table

memo_python = os.environ.get("MEMO_PYTHON")
if not memo_python:
    raise RuntimeError("Set MEMO_PYTHON to the isolated memo worker interpreter")
depths, beta = (0, 1), 1.0
output_root = Path(tempfile.mkdtemp(prefix="memo-python-comparison-"))
print("UMI Python:", sys.version.split()[0])
print("Generated experiment records:", output_root)

UMI Python: 3.11.15
Generated experiment records: /var/folders/h6/g53mxg516n538136hh74630r0000gn/T/memo-python-comparison-xz_4wda9


## The two implementations express the same model

At depth 0, the imagined speaker chooses uniformly among words that truthfully
describe its referent. The listener conditions on the observed word, using a
uniform prior over referents. At depth 1, the speaker prefers words that the
previous listener would interpret correctly:

$$S_d(u\mid r) \propto \mathrm{denotes}(u,r)\exp\{\beta L_{d-1}(r\mid u)\}.$$

The utility here is **success probability**, as in the paper's example; it is not
log probability. Memo represents imagined agents explicitly and performs its own
inference. The Python reference below enumerates and normalizes finite tables
with dictionaries and `math`. It was added for this comparison; it does not
replace the memo inference path.

In [2]:
print("Plain Python reference")
display(Code(inspect.getsource(listener_table), language="python"))
memo_source = Path(memo_package.__file__).with_name("memo_model.py").read_text()
print("Native memo model (executed only by the Python 3.12 worker)")
display(Code(memo_source[memo_source.index("@memo"):memo_source.index("\ndef predict")],
             language="python"))

Plain Python reference


def listener_table(*, depth: int, beta: float = 1.0) -> dict[str, dict[str, float]]:
    """Infer referents with the paper's probability utility and uniform prior.

    At depth zero the speaker chooses uniformly among true utterances.
    At later depths its utility is the previous listener's success probability.
    This deliberately uses Python floats, independently of memo and JAX.
    """
    if type(depth) is not int or depth < 0:
        raise ValueError('depth must be a nonnegative integer')
    if isinstance(beta, bool) or not isinstance(beta, (int, float)):
        raise TypeError('beta must be numeric')
    if not math.isfinite(beta):
        raise ValueError('beta must be finite')
    previous = listener_table(depth=depth - 1, beta=beta) if depth else None
    speaker = {}
    for referent in CHOICES:
        weights = {
            utterance: (0.0 if referent not in MEANINGS[utterance] else
                        1.0 if previous is None else
                        math.exp(beta * previous[utterance][referent]))
            for utterance in UTTERANCES
        }
        total = math.fsum(weights.values())
        speaker[referent] = {word: weight / total for word, weight in weights.items()}

    # The uniform referent prior cancels in Bayes' rule.
    return {
        utterance: {
            referent: speaker[referent][utterance] / math.fsum(
                speaker[other][utterance] for other in CHOICES)
            for referent in CHOICES
        }
        for utterance in UTTERANCES
    }

Native memo model (executed only by the Python 3.12 worker)


@memo
def L[u: U, r: R](beta, depth):
    listener: thinks[
        speaker: chooses(r_true in R, wpp=1),
        speaker: chooses(u_said in U, wpp=
            denotes(u_said, r_true) *
            (1 if depth == 0 else exp(beta * L[u_said, r_true](beta, depth - 1)))
        ),
    ]
    listener: observes [speaker.u_said] is u
    listener: chooses(r_pick in R, wpp=Pr[speaker.r_true == r_pick])
    return Pr[listener.r_pick == r]

In [3]:
native_tables = {}
worker = MemoWorker(python_executable=memo_python)
try:
    for depth in depths:
        native_tables["memo", depth] = np.asarray(worker.predict(
            depth=depth, beta=beta, utterances=list(UTTERANCES)), dtype=np.float64)
        python_predictions = listener_table(depth=depth, beta=beta)
        native_tables["python", depth] = np.asarray([
            [python_predictions[word][choice] for choice in CHOICES]
            for word in UTTERANCES
        ], dtype=np.float64)
    memo_runtime = dict(worker.runtime)
finally:
    worker.close()

rows = [
    {"implementation": implementation, "depth": depth, "utterance": word,
     **dict(zip(CHOICES, values))}
    for (implementation, depth), table in native_tables.items()
    for word, values in zip(UTTERANCES, table)
]
with pd.option_context("display.precision", 17):
    display(pd.DataFrame(rows).set_index(["implementation", "depth", "utterance"]))
print("Memo worker:", memo_runtime)

green square         green circle  \
implementation depth utterance                                             
memo           0     green      0.50000000000000000  0.50000000000000000   
                     pink       0.00000000000000000  0.00000000000000000   
                     square     1.00000000000000000  0.00000000000000000   
                     round      0.00000000000000000  0.50000000000000000   
python         0     green      0.50000000000000000  0.50000000000000000   
                     pink       0.00000000000000000  0.00000000000000000   
                     square     1.00000000000000000  0.00000000000000000   
                     round      0.00000000000000000  0.50000000000000000   
memo           1     green      0.43022581934928894  0.56977415084838867   
                     pink       0.00000000000000000  0.00000000000000000   
                     square     1.00000000000000000  0.00000000000000000   
                     round      0.00000000000000000  0.56977415084838867   
python         1     green      0.43022583707170436  0.56977416292829564   
                     pink       0.00000000000000000  0.00000000000000000   
                     square     1.00000000000000000  0.00000000000000000   
                     round      0.00000000000000000  0.56977416292829564   

                                        pink circle  
implementation depth utterance                       
memo           0     green      0.00000000000000000  
                     pink       1.00000000000000000  
                     square     0.00000000000000000  
                     round      0.50000000000000000  
python         0     green      0.00000000000000000  
                     pink       1.00000000000000000  
                     square     0.00000000000000000  
                     round      0.50000000000000000  
memo           1     green      0.00000000000000000  
                     pink       1.00000000000000000  
                     square     0.00000000000000000  
                     round      0.43022581934928894  
python         1     green      0.00000000000000000  
                     pink       1.00000000000000000  
                     square     0.00000000000000000  
                     round      0.43022583707170436

Memo worker: {'python': '3.12.4', 'python_major_minor': [3, 12], 'memo_version': '1.3.3', 'jax_version': '0.12.0.1', 'jaxlib_version': '0.12.0.1', 'jax_enable_x64': False, 'jax_default_matmul_precision': None, 'devices': [{'platform': 'cpu', 'device_kind': 'cpu', 'id': 0}], 'backend': 'cpu', 'inference_dtype': 'float32'}


## Compare values without hiding numerical differences

Plain Python uses binary64 floating point. Memo uses JAX's configured precision
(the pinned worker defaults to binary32). The same mathematical probabilities
can therefore differ in their last digits. We report the measured difference
between implementations and do not pass it through a tolerance test.

For integration fidelity, each implementation's **native versus UMI** outputs
must instead match exactly under the same execution settings.

In [4]:
differences = pd.DataFrame([
    {"depth": depth,
     "exactly_equal_across_implementations": np.array_equal(
         native_tables["memo", depth], native_tables["python", depth]),
     "maximum_absolute_difference": float(np.max(np.abs(
         native_tables["memo", depth] - native_tables["python", depth])))}
    for depth in depths
]).set_index("depth")
with pd.option_context("display.precision", 17):
    display(differences)

,exactly_equal_across_implementations,maximum_absolute_difference
depth,,
0,True,0.00000000000000000
1,False,0.00000001772241542


## Present the same experiment through UMI

Both subjects declare `instruction` and `text` inputs and `behavior` output.
Only their inference implementations differ. The memo subject uses its isolated
worker; this small Python subject calls the reference function above. Nested
imagined agents remain internal to each model.

`SessionProtocol` owns resets before and after each run. Events have `t_ms=None`
because this task has no physical clock. The buffered session collects labeled
probability responses into a `BehavioralAssembly`.

In [5]:
class PythonRSASubject(Subject):
    identifier = "python-rsa"
    in_channels = {"instruction", "text"}
    required_channels = {"instruction", "text"}
    out_channels = {"behavior"}

    def __init__(self, *, depth: int, beta: float) -> None:
        self.depth, self.beta = depth, beta
        self.identifier = f"python-rsa-depth-{depth}-beta-{beta:g}"

    def interact(self, session: Session) -> None:
        if tuple(session.requested_output_channels or ()) != ("behavior",):
            raise ValueError("Request the behavior output channel")
        instruction_seen = False
        table = None
        while (event := session.next_input()) is not None:
            if event.channel == "instruction":
                if instruction_seen or event.payload != SCENE:
                    raise ValueError("Expected one three-object instruction")
                instruction_seen = True
                continue
            if event.channel != "text":
                continue
            if not instruction_seen or event.payload not in UTTERANCES:
                raise ValueError("Expected the task instruction followed by a known utterance")
            if table is None:
                table = listener_table(depth=self.depth, beta=self.beta)
            session.respond(StreamEvent(
                channel="behavior",
                payload=np.asarray([table[event.payload][choice] for choice in CHOICES]),
                t_ms=event.t_ms,
                meta={"stimulus_id": event.meta["stimulus_id"], "label_set": CHOICES,
                      "provenance": {"implementation": "plain Python RSA",
                                     "depth": self.depth, "beta": self.beta,
                                     "python": sys.version.split()[0],
                                     "numeric_type": "binary64 Python float"}},
            ))
        if not instruction_seen or table is None:
            raise ValueError("Expected the three-object instruction and utterances")

In [6]:
@contextmanager
def session_factory(_trial: Trial) -> Iterator[BufferedSession]:
    events = [StreamEvent(channel="instruction", payload=SCENE, t_ms=None)]
    events.extend(StreamEvent(
        channel="text", payload=word, t_ms=None, meta={"stimulus_id": word},
    ) for word in UTTERANCES)
    yield BufferedSession(inputs=events, requested_output_channels=("behavior",))

experiment_protocol = SessionProtocol(
    identifier="three-object-rsa", session_factory=session_factory,
    trials=("four-utterances",), input_channels=("instruction", "text"),
    output_channels=("behavior",),
    metadata={"utterances": UTTERANCES, "choices": CHOICES, "scene": SCENE},
)

In [7]:
runs = {}
checks = []
recorded_inputs = []
for implementation in ("memo", "python"):
    for depth in depths:
        subject = (MemoSubject(config=MemoConfig(depth=depth, beta=beta),
                               python_executable=memo_python)
                   if implementation == "memo" else PythonRSASubject(depth=depth, beta=beta))
        result = Experiment(
            subject=subject, protocol=experiment_protocol, tools=(RecordInputsOutputs(),),
            output_dir=output_root / f"{implementation}-depth-{depth}",
            metadata={"comparison": "model configuration", "implementation": implementation,
                      "depth": depth, "beta": beta},
        ).run()
        behavior = result.value[0]
        assert np.array_equal(behavior.values, native_tables[implementation, depth])
        assert tuple(behavior.stimulus_id.values) == UTTERANCES
        assert tuple(behavior.choice.values) == CHOICES
        recorded_inputs.append([event["payload"] for event in read_events(result.directory)
                                if event["kind"] == "input"])
        runs[implementation, depth] = result
        checks.append({"implementation": implementation, "depth": depth,
                       "native_equals_UMI_exactly": True})
assert all(events == recorded_inputs[0] for events in recorded_inputs)
display(pd.DataFrame(checks).set_index(["implementation", "depth"]))
print("All four runs received identical recorded input events.")

native_equals_UMI_exactly
implementation depth                           
memo           0                           True
               1                           True
python         0                           True
               1                           True

All four runs received identical recorded input events.


In [8]:
events = read_events(runs["memo", 1].directory)
outputs = [event["payload"] for event in events if event["kind"] == "output"]
display(pd.DataFrame([
    {"utterance": event.meta["stimulus_id"], **dict(zip(event.meta["label_set"], event.payload))}
    for event in outputs
]).set_index("utterance"))
print("Provenance attached to each memo response:")
print(json.dumps(outputs[0].meta["provenance"], indent=2))
print("Raw recordings contain JSON manifests/events and referenced NumPy arrays.")
print("They are generated locally rather than versioned in the repository.")

,green square,green circle,pink circle
utterance,,,
green,0.430226,0.569774,0.000000
pink,0.000000,0.000000,1.000000
square,1.000000,0.000000,0.000000
round,0.000000,0.569774,0.430226


Provenance attached to each memo response:
{
  "implementation": "memo Rational Speech Acts example",
  "memo_source_reviewed_revision": "d8aac635ee53483e8e0311b85f6898ef37710a26",
  "memo_distribution": "memo-lang",
  "implementation_sha256": {
    "model.py": "af04e6da2363efbed2fc82daf7a7c82e86d287f899348ff7f43340ac50e3e1bc",
    "worker.py": "dfd0bdcd3ddf4b289d0ffd599282564ff667140ce5dc2a59a7f500fe9cf042b1",
    "memo_worker.py": "68c0baa8e9099c4f33e058d05b96070b5980038c1e5ed05fce980af7c3dd3036",
    "memo_model.py": "4e6d00ae8725c25a9950b9bb55fb8fbc04c915c03905319d960f70a8f8e6e806"
  },
  "host_numpy": "1.26.4",
  "output_dtype": "float64",
  "reasoning_depth": 1,
  "beta": 1.0,
  "worker_python": "/private/tmp/memo-rsa-py312/bin/python",
  "host_python": "3.11.15",
  "umi_base_revision": "abf74bd55a48bd6585d08ccd3320dd30a1e9ceca",
  "umi_runtime_revision": "b8880276242296c6857363777b4bd12a02ab5213",
  "pinned_revisions": {
    "core": "15dcc3c6cbfea16565052abd2663b8de4bf1b203",
  

In [9]:
replay_checks = []
for implementation in ("memo", "python"):
    source = runs[implementation, 1]
    subject = (MemoSubject(config=MemoConfig(depth=1, beta=beta),
                           python_executable=memo_python)
               if implementation == "memo" else PythonRSASubject(depth=1, beta=beta))
    replay = Experiment(
        subject=subject, protocol=replay_sessions(source.directory),
        tools=(RecordInputsOutputs(),),
        output_dir=output_root / f"{implementation}-depth-1-replay",
        metadata={"mode": "saved inputs; model rerun"},
    ).run()
    comparison = compare_outputs(source.directory, replay.directory)
    assert comparison["equal"]
    replay_checks.append({"implementation": implementation, "exact_replay": comparison["equal"]})
display(pd.DataFrame(replay_checks).set_index("implementation"))
print("Replay re-presents saved inputs and reruns inference; it does not emit saved responses.")

,exact_replay
implementation,
memo,True
python,True


Replay re-presents saved inputs and reruns inference; it does not emit saved responses.


In [10]:
display(pd.DataFrame([
    {"implementation": implementation, "depth": depth,
     "P(green circle | green)": table[0, 1]}
    for (implementation, depth), table in native_tables.items()
]).set_index(["implementation", "depth"]))
print("Full local experiment records:", output_root)

,,P(green circle | green)
implementation,depth,
memo,0,0.500000
python,0,0.500000
memo,1,0.569774
python,1,0.569774


Full local experiment records: /var/folders/h6/g53mxg516n538136hh74630r0000gn/T/memo-python-comparison-xz_4wda9


At depth 0, both listeners split ambiguous **green** equally. At depth 1, both
favor the green circle because a speaker intending the square could have used
**square** unambiguously. The displayed differences between implementations are
numerical; each implementation preserved its own native probabilities exactly
through UMI and recorded-input replay.

Depth is a model configuration, not a reversible intervention or a measure of
human cognitive ability. No human observations were invented or fitted. Memo's
agent syntax, finite enumerative inference, and JAX compilation remain inside
the memo runtime; Python explicitly performs the table operations. Worker
resources were closed and UMI resets were owned by the session protocol. Rerun
all cells to create a fresh output directory.